In [ ]:
!pip install -q diffusers transformers accelerate gradio

import torch
from diffusers import StableDiffusionPipeline
import gradio as gr

model_id = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe = pipe.to("cuda")

def generate_image(prompt):
    image = pipe(prompt).images[0]
    return image

samples = [
    ["A futuristic city with flying cars, neon lights, highly detailed"],
    ["A cute hamster wearing a wizard hat, digital art style"],
    ["Breathtaking landscape of a hidden waterfall in a tropical forest"],
    ["An oil painting of a cat playing the piano in a cozy room"],
    ["Cyberpunk robot garden with metallic flowers, 4k resolution"]
]

demo = gr.Interface(
    fn=generate_image,
    inputs=gr.Textbox(label="그리려는 이미지를 영어로 설명하세요", placeholder="An astronaut riding a horse in space..."),
    outputs=gr.Image(label="생성된 이미지"),
    title="🎨 AI 화가 데모 (Stable Diffusion)",
    description="문장을 입력하면 AI가 그림을 그려줍니다.",
    examples=samples
)

if __name__ == "__main__":
    demo.launch(share=True)

In [ ]:
import torch
from transformers import DetrImageProcessor, DetrForObjectDetection
from PIL import Image, ImageDraw
import gradio as gr

processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")

def detect_objects(image):
    inputs = processor(images=image, return_tensors="pt")
    outputs = model(**inputs)

    target_sizes = torch.tensor([image.size[::-1]])
    results = processor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=0.9)[0]

    draw = ImageDraw.Draw(image)
    for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
        box = [round(i, 2) for i in box.tolist()]
        draw.rectangle(box, outline="red", width=3)
        draw.text((box[0], box[1]), f"{model.config.id2label[label.item()]}: {round(score.item(), 2)}", fill="red")

    return image

samples = [
    ["https://images.unsplash.com/photo-1574353260424-97ee3385ea0e?w=500&auto=format&fit=crop&q=60&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxzZWFyY2h8Mnx8JUVCJThGJTg0JUVCJUExJTlDJTIwJUVEJTk2JTg5JUVDJTlEJUI4JTIwJUVDJTlFJTkwJUVCJThGJTk5JUVDJUIwJUE4fGVufDB8fDB8fHww$0"],
    ["https://images.unsplash.com/photo-1544032527-042957c6f7ce?w=500&auto=format&fit=crop&q=60&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxzZWFyY2h8M3x8JUVCJThGJTg0JUVCJUExJTlDJTIwJUVEJTk2JTg5JUVDJTlEJUI4JTIwJUVDJTlFJTkwJUVCJThGJTk5JUVDJUIwJUE4fGVufDB8fDB8fHww$0"],
    ["https://images.unsplash.com/photo-1611147533125-9ca445f32036?w=500&auto=format&fit=crop&q=60&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxzZWFyY2h8MTB8fCVFQiU4RiU4NCVFQiVBMSU5QyUyMCVFRCU5NiU4OSVFQyU5RCVCOCUyMCVFQyU5RSU5MCVFQiU4RiU5OSVFQyVCMCVBOHxlbnwwfHwwfHx8MA%3D%3D$0"],
    ["https://images.unsplash.com/photo-1736000351052-aed67e7372dc?w=500&auto=format&fit=crop&q=60&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxzZWFyY2h8MTV8fCVFQiU4RiU4NCVFQiVBMSU5QyUyMCVFRCU5NiU4OSVFQyU5RCVCOCUyMCVFQyU5RSU5MCVFQiU4RiU5OSVFQyVCMCVBOHxlbnwwfHwwfHx8MA%3D%3D$0"],
    ["https://images.unsplash.com/photo-1713388889443-9f5e05f91760?w=500&auto=format&fit=crop&q=60&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxzZWFyY2h8MTZ8fCVFQiU4RiU4NCVFQiVBMSU5QyUyMCVFQyU4QiVBMCVFRCU5OCVCOCVFQiU5MyVCMSUyMCVFQyU5RSU5MCVFQiU4RiU5OSVFQyVCMCVBOCUyMCVFRCU5NiU4OSVFQyU5RCVCOHxlbnwwfHwwfHx8MA%3D%3D$0"]
]

demo_detection = gr.Interface(
    fn=detect_objects,
    inputs=gr.Image(type="pil", label="도로 사진을 업로드하세요"),
    outputs=gr.Image(type="pil", label="인식 결과"),
    title="🚗 자율주행 Object Detection 데모",
    description="이미지 내의 차량, 보행자, 신호등 등을 AI가 인식합니다.",
    examples=samples
)

demo_detection.launch(share=True)

In [ ]:
!pip install -q "rembg[cpu]"
from rembg import remove
import gradio as gr
from PIL import Image

def remove_background(input_image):
    if input_image is None: return None
    return remove(input_image)

samples = [
    ["https://images.unsplash.com/photo-1529626455594-4ff0802cfb7e?q=80&w=1000"],
    ["https://images.unsplash.com/photo-1542291026-7eec264c27ff?q=80&w=1000"],
    ["https://images.unsplash.com/photo-1523275335684-37898b6baf30?q=80&w=1000"],
    ["https://images.unsplash.com/photo-1505740420928-5e560c06d30e?q=80&w=1000"],
    ["https://images.unsplash.com/photo-1574634534894-89d7576c8259?q=80&w=1000"]
]

demo_rembg = gr.Interface(
    fn=remove_background,
    inputs=gr.Image(type="pil", label="배경을 지울 사진"),
    outputs=gr.Image(type="pil", label="배경이 제거된 결과"),
    title="✂️ AI 누끼 따기 (Background Remover)",
    description="사진을 업로드하면 배경을 자동으로 투명하게 만들어줍니다.",
    examples=samples
)

demo_rembg.launch(share=True)

In [ ]:
import gradio as gr
from transformers import pipeline

food_classifier = pipeline("image-classification", model="google/vit-base-patch16-224")

def classify_food(image):
    if image is None: return None
    predictions = food_classifier(image)
    return {pred['label']: pred['score'] for pred in predictions}

samples = [
    ["https://images.unsplash.com/photo-1513104890138-7c749659a591?auto=format&fit=crop&q=80&w=1000"],
    ["https://images.unsplash.com/photo-1568901346375-23c9450c58cd?auto=format&fit=crop&q=80&w=1000"],
    ["https://images.unsplash.com/photo-1546069901-ba9599a7e63c?auto=format&fit=crop&q=80&w=1000"],
    ["https://images.unsplash.com/photo-1551183053-bf91a1d81141?auto=format&fit=crop&q=80&w=1000"],
    ["https://images.unsplash.com/photo-1600891964599-f61ba0e24092?auto=format&fit=crop&q=80&w=1000"]
]

demo_food = gr.Interface(
    fn=classify_food,
    inputs=gr.Image(type="pil", label="음식 사진을 올려주세요"),
    outputs=gr.Label(num_top_classes=3, label="분석 결과"),
    title="🍕 AI 음식 인식기",
    description="어떤 음식인지 상위 3개 후보를 알려줍니다.",
    examples=samples
)

demo_food.launch(share=True)

In [ ]:
import gradio as gr
import torch
from PIL import Image

try:
    model = torch.hub.load("AK391/animegan2-pytorch:main", "generator", pretrained="face_paint_512_v2", trust_repo=True)
    face2paint = torch.hub.load("AK391/animegan2-pytorch:main", "face2paint", size=512, trust_repo=True)
    device = "cuda"
    model.to(device).eval()
except Exception as e:
    print(f"모델 로드 중 오류 발생: {e}")

def cartoonize(image):
    if image is None: return None
    
    try:
        input_img = image.convert("RGB")
        
        with torch.no_grad():
            output = face2paint(model, input_img, side_by_side=False, device=device)
        
        return output
        
    except Exception as e:
        return f"처리 중 오류가 발생했습니다: {str(e)}"

samples = [
    ["https://images.unsplash.com/photo-1541963463532-d68292c34b19?auto=format&fit=crop&q=80&w=1000"], 
    ["https://images.unsplash.com/photo-1511367461989-f85a21fda167?auto=format&fit=crop&q=80&w=1000"], 
    ["https://images.unsplash.com/photo-1464822759023-fed622ff2c3b?auto=format&fit=crop&q=80&w=1000"], 
    ["https://images.unsplash.com/photo-1441974231531-c6227db76b6e?auto=format&fit=crop&q=80&w=1000"], 
    ["https://images.unsplash.com/photo-1501785888041-af3ef285b470?auto=format&fit=crop&q=80&w=1000"]
]

demo_cartoon = gr.Interface(
    fn=cartoonize,
    inputs=gr.Image(type="pil", label="원본 사진"),
    outputs=gr.Image(type="pil", label="애니메이션 스타일 결과"),
    title="🎨 AI 애니메이션 필터",
    description="사진을 고퀄리티 애니메이션 스타일로 변환합니다.",
    examples=samples
)

if __name__ == "__main__":
    demo_cartoon.launch(share=True)